# Consensus Principle Selection

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import time
import sys
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
DATA_PATH = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_ai_principles_th=0.7_summarized_test.json") 

TOP_K_SUMMARIES = 20

TRESHOLD = "0.7_test"

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
def embed_texts(texts):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            resp = client.embeddings.create(
                model="text-embedding-3-small",
                input=texts,
            )
            vectors = [item.embedding for item in resp.data]
            return np.array(vectors, dtype=float)
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)

In [4]:
# Load clustered principles
with DATA_PATH.open("r", encoding="utf-8") as f:
    principles_data = json.load(f)

summaries = []
summary_cluster_ids = []
originals = []

for idx, item in enumerate(principles_data):
    cluster_id = item.get("cluster_id")

    summarized_principle = item.get("summarized_principle").strip()
    principle_list = item.get("original_principles", [])

    if summarized_principle:
        summaries.append(summarized_principle)
        summary_cluster_ids.append(cluster_id)

    for p in principle_list:
        p = (p or "").strip()
        if p:
            originals.append(p)

print(f"#summaries: {len(summaries)}")
print(f"#originals: {len(originals)}")


#summaries: 25
#originals: 1500


In [5]:
emb_orig = embed_texts(originals)  
emb_summaries = embed_texts(summaries)  

N, d = emb_orig.shape if emb_orig.size else (0, 0)
M, d_s = emb_summaries.shape if emb_summaries.size else (0, 0)

print(f"emb_orig shape: {emb_orig.shape}")
print(f"emb_summaries shape: {emb_summaries.shape}")

emb_orig shape: (1500, 1536)
emb_summaries shape: (25, 1536)


In [6]:
global_msd = []  

for j in range(M):
    s_j = emb_summaries[j] 
    diff = emb_orig - s_j   
    msd_j = (diff ** 2).mean() 
    global_msd.append(float(msd_j))

global_msd = np.array(global_msd)
global_msd[:10]  

array([0.0008546 , 0.00059387, 0.00084817, 0.00083633, 0.00086787,
       0.00084966, 0.0007458 , 0.00079891, 0.00076121, 0.00093634])

In [7]:
idx_sorted = np.argsort(global_msd)

rows = []
for rank, j in enumerate(idx_sorted, start=1):
    rows.append(
        {
            "rank": rank,
            "cluster_id": summary_cluster_ids[j],
            "summary_text": summaries[j],
            "global_msd": float(global_msd[j]),
        }
    )

df_summaries = pd.DataFrame(rows)
df_summaries.head(TOP_K_SUMMARIES)

,rank,cluster_id,summary_text,global_msd
0,1,1,"The AI should provide accurate, unbiased, and ...",0.000594
1,2,6,AI systems should consider and respect individ...,0.000746
2,3,8,The AI should be capable of general problem so...,0.000761
3,4,11,"AI systems should embody empathy, emotional un...",0.000780
4,5,7,"Provide accurate, unbiased, and straightforwar...",0.000799
5,6,10,AI systems should consistently provide reliabl...,0.000821
6,7,15,AI systems should avoid making claims or decis...,0.000828
7,8,3,"Be knowledgeable, responsive to instructions, ...",0.000836
8,9,2,Provide helpful and comprehensive answers by o...,0.000848
9,10,5,"Communicate thoughtfully, with intelligence, k...",0.000850


In [8]:
output_json_path = Path(f"consensus_principle_selection_th={TRESHOLD}.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

Saved JSON to: consensus_principle_selection_th=0.7_test.json


In [9]:
top = df_summaries.head(TOP_K_SUMMARIES)
for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Global MSD:", row["global_msd"])
    print("Summary:\n", row["summary_text"])
    print("-" * 80)

Rank: 1
Cluster: 1
Global MSD: 0.00059386965871747
Summary:
 The AI should provide accurate, unbiased, and objective information in a clear, concise, and professional manner while maintaining a friendly and compassionate tone, demonstrating an understanding of nuance, and delivering creative or detailed responses as appropriate.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 6
Global MSD: 0.0007457976001470217
Summary:
 AI systems should consider and respect individual user preferences and beliefs when providing responses.
--------------------------------------------------------------------------------
Rank: 3
Cluster: 8
Global MSD: 0.0007612098737216514
Summary:
 The AI should be capable of general problem solving and data handling, provide assistance tailored to varying skill levels and user interests, and support diverse interactive activities.
--------------------------------------------------------------------------------
Rank: 4
